In [ ]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
import gradio as gr
import sqlite3
import pandas as pd

In [ ]:
# The usual start

load_dotenv(override=True)
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

# Pushover Test

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
def record_unknown_question(question):

    push(
        f"Unknown basketball question:\n{question}"
    )

    return {
        "recorded": "ok"
    }

# Data cleaning & SQLite

In [ ]:
DB_PATH = "knicks_coach.db"


def create_database():

    conn = sqlite3.connect(DB_PATH)

    traditional_df = pd.read_csv("data/traditional_stats_q1.csv")
    advanced_df = pd.read_csv("data/advanced_stats_q1.csv")
    matchups_df = pd.read_csv("data/prev_matchups.csv")

    # clean columns
    traditional_df.columns = [
        c.strip().lower()
        .replace(" ", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_")
        for c in traditional_df.columns
    ]

    advanced_df.columns = [
        c.strip().lower()
        .replace(" ", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_")
        for c in advanced_df.columns
    ]

    matchups_df.columns = [
        c.strip().lower()
        .replace(" ", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_")
        for c in matchups_df.columns
    ]

    # save to sqlite
    traditional_df.to_sql("traditional_stats", conn, if_exists="replace", index=False)
    advanced_df.to_sql("advanced_stats", conn, if_exists="replace", index=False)
    matchups_df.to_sql("matchups", conn, if_exists="replace", index=False)

    conn.commit()
    conn.close()

    print("Database created successfully")

In [ ]:
if not os.path.exists(DB_PATH):
    create_database()

# Connect to DB

In [ ]:
DB_PATH = "knicks_coach.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

# Build the first tool

In [ ]:
def get_game_context():

    conn = get_connection()

    traditional = pd.read_sql_query(
        "SELECT * FROM traditional_stats",
        conn
    )

    advanced = pd.read_sql_query(
        "SELECT * FROM advanced_stats",
        conn
    )

    matchups = pd.read_sql_query(
        "SELECT * FROM matchups",
        conn
    )

    conn.close()

    return {
        "traditional_stats":
            traditional.to_dict("records"),

        "advanced_stats":
            advanced.to_dict("records"),

        "matchups":
            matchups.to_dict("records")
    }

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": """
    Use this tool whenever a question
    cannot be answered using the
    available statistics and matchup data.
    """,
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

# Create the Tool Schema

In [ ]:
get_game_context_json = {
    "name": "get_game_context",
    "description": """
    Retrieve all available game information.

    Includes:
    - Q1 traditional stats
    - Q1 advanced stats
    - historical matchup data

    Use this information to recommend
    lineups and strategies for the
    beginning of Q2.
    """,
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {
        "type": "function",
        "function": get_game_context_json
    },
    {
        "type": "function",
        "function": record_unknown_question_json
    }
]

# Create the Coach Prompt

In [ ]:
system_prompt = """
You are an NBA assistant coach for the New York Knicks.

Your job is to help the coaching staff
make decisions for the beginning of Q2.

You have access to:

1. Traditional Q1 player statistics
2. Advanced Q1 player statistics
3. Historical matchup data

When you need data,
use the get_game_context tool.

Your responsibilities include:

- Recommending the best lineup for Q2
- Identifying favorable matchups
- Suggesting offensive adjustments
- Suggesting defensive adjustments
- Recommending player substitutions
- Answering questions about the game

Always support your recommendations
with data from the tool.

If a question cannot be answered
using the available Q1 statistics,
advanced statistics,
or matchup information,
use the record_unknown_question tool.

Do not invent data.
Do not guess.
If you don't find the player name in the data, use the record_unknown_question tool.
"""

In [ ]:
TOOLS_MAP = {
    "get_game_context": get_game_context,
    "record_unknown_question": record_unknown_question
}

In [ ]:
def handle_tool_calls(tool_calls):

    results = []

    for tool_call in tool_calls:

        tool_name = tool_call.function.name

        arguments = json.loads(
            tool_call.function.arguments
        )

        print(
            f"Tool called: {tool_name}"
        )

        tool = TOOLS_MAP.get(tool_name)

        result = (
            tool(**arguments)
            if tool else {}
        )

        results.append(
            {
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id
            }
        )

    return results

# Use an UI chat (Gradio) and Build LLM using GPT-mini

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    # Resolve tool calls first (non-streaming), then stream the final reply
    while True:
        response = openai.chat.completions.create(
            model="gpt-4o-mini", messages=messages, tools=tools
        )
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            results = handle_tool_calls(assistant_message.tool_calls)
            messages.append(assistant_message)
            messages.extend(results)
        else:
            break

    stream = openai.chat.completions.create(
        model="gpt-4o-mini", messages=messages, stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="🏀 Knicks Q2 Coaching Assistant",
    description="""
    AI assistant for lineup and strategy decisions.

    Uses:
    • Q1 Traditional Stats
    • Q1 Advanced Stats
    • Historical Matchups

    Ask questions about:
    • Best Q2 lineup
    • Matchup advantages
    • Offensive strategy
    • Defensive adjustments
    • Player rotations
    """
)

demo.launch()